#### Business Overview (Total Revenue, Orders, and Customers)
---

In [1]:
# Database connection setup and verification
import pandas as pd
from sqlalchemy import create_engine

db_user: str = "root"
db_password: str = "1234"
db_host: str = "localhost"
db_port: str = "3306"
db_name: str = "ecommerce_db"

db_url: str = f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
db_engine = create_engine(db_url)

# Verify active database connection
df_check: pd.DataFrame = pd.read_sql("SELECT DATABASE() AS active_db;", con=db_engine)
display(df_check)


,active_db
0,ecommerce_db


In [2]:
# Query 1: Platform macro KPIs (total orders, gross revenue, and average order value)
query_01: str = """
SELECT
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(payment_value), 2) AS total_revenue,
    ROUND(AVG(payment_value), 2) AS avg_order_value
FROM order_payments;
"""

df_kpi: pd.DataFrame = pd.read_sql(query_01, con=db_engine)
display(df_kpi)


,total_orders,total_revenue,avg_order_value
0,99440,16008872.12,154.1


In [3]:
# Query 2: Monthly revenue and order volume trends for delivered orders
query_02: str = """
SELECT
    SUBSTRING(o.order_purchase_timestamp, 1, 7) AS order_month,
    ROUND(SUM(p.payment_value), 2) AS monthly_revenue,
    COUNT(DISTINCT o.order_id) AS order_count
FROM orders o
JOIN order_payments p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY order_month
ORDER BY order_month;
"""

df_monthly_trend: pd.DataFrame = pd.read_sql(query_02, con=db_engine)
display(df_monthly_trend)


,order_month,monthly_revenue,order_count
0,2016-10,46566.71,265
1,2016-12,19.62,1
2,2017-01,127545.67,750
3,2017-02,271298.65,1653
4,2017-03,414369.39,2546
5,2017-04,390952.18,2303
6,2017-05,567066.73,3546
7,2017-06,490225.60,3135
8,2017-07,566403.93,3872
9,2017-08,646000.61,4193


In [4]:
# Query 3: Top 10 product categories by total sales revenue
query_03: str = """
SELECT 
    p.product_category_name_english AS category,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders o ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 10;
"""

df_top_categories: pd.DataFrame = pd.read_sql(query_03, con=db_engine)
display(df_top_categories)


,category,total_orders,total_revenue
0,health_beauty,8647,1233131.72
1,watches_gifts,5495,1166176.98
2,bed_bath_table,9272,1023434.76
3,sports_leisure,7530,954852.55
4,computers_accessories,6530,888724.61
5,furniture_decor,6307,711927.69
6,housewares,5743,615628.69
7,cool_stuff,3559,610204.10
8,auto,3810,578966.65
9,toys,3804,471286.48


In [5]:
# Query 4: Customer geographic concentration across top 10 states
query_04: str = """
SELECT 
    c.customer_state,
    COUNT(DISTINCT c.customer_unique_id) AS total_customers,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_state
ORDER BY total_orders DESC
LIMIT 10;
"""

df_customer_states: pd.DataFrame = pd.read_sql(query_04, con=db_engine)
display(df_customer_states)


,customer_state,total_customers,total_orders
0,SP,40302,41746
1,RJ,12384,12852
2,MG,11259,11635
3,RS,5277,5466
4,PR,4882,5045
5,SC,3534,3637
6,BA,3277,3380
7,DF,2075,2140
8,ES,1964,2033
9,GO,1952,2020


In [6]:
# Query 5: Payment method transaction counts and installment averages
query_05: str = """
SELECT 
    payment_type,
    COUNT(*) AS transaction_count,
    ROUND(SUM(payment_value), 2) AS total_payment_value,
    ROUND(AVG(payment_installments), 1) AS avg_installments
FROM order_payments
GROUP BY payment_type
ORDER BY transaction_count DESC;
"""

df_payments_analysis: pd.DataFrame = pd.read_sql(query_05, con=db_engine)
display(df_payments_analysis)


,payment_type,transaction_count,total_payment_value,avg_installments
0,credit_card,76795,12542084.19,3.5
1,boleto,19784,2869361.27,1.0
2,voucher,5775,379436.87,1.0
3,debit_card,1529,217989.79,1.0
4,other,3,0.00,1.0


In [7]:
# Query 6: Average delivery latency in days by customer state
query_06: str = """
SELECT 
    c.customer_state,
    ROUND(AVG(DATEDIFF(o.order_delivered_customer_date, o.order_purchase_timestamp)), 1) AS avg_delivery_days,
    COUNT(DISTINCT o.order_id) AS total_delivered_orders
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
    AND o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY avg_delivery_days DESC
LIMIT 10;
"""

df_delivery_latency: pd.DataFrame = pd.read_sql(query_06, con=db_engine)
display(df_delivery_latency)


,customer_state,avg_delivery_days,total_delivered_orders
0,RR,29.3,41
1,AP,27.2,67
2,AM,26.4,145
3,AL,24.5,397
4,PA,23.7,946
5,SE,21.5,335
6,MA,21.5,717
7,CE,21.2,1279
8,AC,21.0,80
9,PB,20.4,517


In [8]:
# Query 7: Top 10 merchant sellers by fulfilled gross revenue (CTE)
query_07: str = """
WITH seller_summary AS (
    SELECT 
        oi.seller_id,
        s.seller_state,
        COUNT(DISTINCT oi.order_id) AS orders_handled,
        ROUND(SUM(oi.price), 2) AS gross_revenue
    FROM order_items oi
    JOIN sellers s ON oi.seller_id = s.seller_id
    GROUP BY oi.seller_id, s.seller_state
)
SELECT 
    seller_id,
    seller_state,
    orders_handled,
    gross_revenue
FROM seller_summary
ORDER BY gross_revenue DESC
LIMIT 10;
"""

df_top_sellers: pd.DataFrame = pd.read_sql(query_07, con=db_engine)
display(df_top_sellers)


,seller_id,seller_state,orders_handled,gross_revenue
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,1132,229472.63
1,53243585a1d6dc2643021fd1853d8905,BA,358,222776.05
2,4a3ca9315b744ce9f8e9374361493884,SP,1806,200472.92
3,fa1c13f2614d7b5c4749cbc52fecda94,SP,585,194042.03
4,7c67e1448b00f6e969d365cea6b010ab,SP,982,187923.89
5,7e93a43ef30c4f03f38b393420bc753a,SP,336,176431.87
6,da8622b14eb17ae2831f4ac5b9dab84a,SP,1314,160236.57
7,7a67c85e85bb2ce8582c35f2203ad736,SP,1160,141745.53
8,1025f0e2d44d7041d6cf58b6550e0bfa,SP,915,138968.55
9,955fee9216a65b617aa5c0531780ce60,SP,1287,135171.70


In [9]:
# Query 8: Top 3 products per category by revenue (Window Function)
query_08: str = """
WITH ranked_products AS (
    SELECT 
        p.product_category_name_english AS category,
        oi.product_id,
        ROUND(SUM(oi.price), 2) AS product_revenue,
        DENSE_RANK() OVER (
            PARTITION BY p.product_category_name_english 
            ORDER BY SUM(oi.price) DESC
        ) AS category_rank
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY category, oi.product_id
)
SELECT 
    category,
    product_id,
    product_revenue,
    category_rank
FROM ranked_products
WHERE category_rank <= 3
ORDER BY category, category_rank;
"""

df_ranked_products: pd.DataFrame = pd.read_sql(query_08, con=db_engine)
display(df_ranked_products.head(15))


,category,product_id,product_revenue,category_rank
0,agro_industry_and_commerce,11250b0d4b709fee92441c5f34122aed,9111.00,1
1,agro_industry_and_commerce,423a6644f0aa529e8828ff1f91003690,8043.00,2
2,agro_industry_and_commerce,672e757f331900b9deea127a2a7b79fd,6885.00,3
3,air_conditioning,12485f9cdebb6ca179826ede539554ad,3899.91,1
4,air_conditioning,83ca77d87b187321faaee535adbce26c,2469.85,2
5,air_conditioning,0e34187d4312b97b5e698836d28ed040,2193.00,3
6,art,4fe644d766c7566dbc46fb851363cb3b,10803.72,1
7,art,1bdf5e6731585cf01aa8169c7028d6ad,6499.00,2
8,art,cbf96c04205dc933b89e025748c2a057,487.50,3
9,arts_and_craftmanship,b9976e9c22fb1540bd71d1bcd2989475,641.45,1


In [10]:
# Query 9: Customer retention: repeat buyers vs one-time shoppers
query_09: str = """
WITH customer_order_counts AS (
    SELECT 
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS total_orders
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
)
SELECT 
    CASE 
        WHEN total_orders > 1 THEN 'Repeat Customer' 
        ELSE 'One-Time Customer' 
    END AS customer_type,
    COUNT(*) AS customer_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customer_order_counts), 2) AS percentage
FROM customer_order_counts
GROUP BY customer_type;
"""

df_retention: pd.DataFrame = pd.read_sql(query_09, con=db_engine)
display(df_retention)


,customer_type,customer_count,percentage
0,One-Time Customer,93099,96.88
1,Repeat Customer,2997,3.12


In [11]:
# Query 10: Impact of delivery delay on customer review ratings
query_10: str = """
SELECT 
    CASE 
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 'Delayed Delivery'
        ELSE 'On-Time Delivery'
    END AS delivery_status,
    COUNT(DISTINCT o.order_id) AS order_count,
    ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
    AND o.order_delivered_customer_date IS NOT NULL
GROUP BY delivery_status;
"""

df_delivery_impact: pd.DataFrame = pd.read_sql(query_10, con=db_engine)
display(df_delivery_impact)


,delivery_status,order_count,avg_review_score
0,Delayed Delivery,7661,2.57
1,On-Time Delivery,88163,4.29
